# M2 — chuẩn hóa dữ liệu M1

Mở notebook bằng Google Colab, đổi `RANGES` ở cell cấu hình rồi chọn **Runtime → Run all**. Notebook tự tải code M2 từ GitHub; không cần chép project lên Drive. Tài khoản Google cần có quyền đọc thư mục checkpoint M1 (mỗi range cần có `range_N_dev` với các file batch). Các phiên bản không có LaTeX được ghi nhận và bỏ qua phần chuẩn hóa.


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()


In [ ]:
from pathlib import Path
import os, subprocess, sys

# Chỉ cần đổi danh sách range muốn chạy (từ 1 đến 5).
RANGES = [1, 2]  # Ví dụ: [3] hoặc [3, 4, 5]

# Giữ nguyên nếu dùng thư mục checkpoint M1 chung của nhóm.
M1_PARENT_ID = '1LVKMyNXHzPgOQS3tufc5-PIDHzcMZqxu'
WORKERS = min(2, os.cpu_count() or 1)
OUTPUT = Path('/content/drive/MyDrive/M2_STANDARDIZED_REV14')
# Lưu workspace trên Drive để tiếp tục sau khi Colab ngắt.
WORKSPACE = Path('/content/drive/MyDrive/M2_STANDARDIZED_WORKSPACE_REV14')

# Cố định commit đã có code chuẩn hóa Rev14 để mọi người chạy cùng phiên bản.
REPO_URL = 'https://github.com/tuandatedu/arXir_project.git'
CODE_COMMIT = '21aa3f3c32cbe528b02ca5d45a7475ae3099155f'
PROJECT_ROOT = Path('/content/arXir_m2_code')

if not RANGES or any(type(n) is not int or n not in range(1, 6) for n in RANGES) or len(set(RANGES)) != len(RANGES):
    raise ValueError('RANGES phải gồm các số khác nhau từ 1 đến 5, ví dụ [3, 4].')


In [ ]:
# Tải code vào máy Colab; dữ liệu và kết quả vẫn nằm trên Drive.
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    if not (PROJECT_ROOT / '.git').is_dir():
        raise RuntimeError(f'{PROJECT_ROOT} đã tồn tại nhưng không phải Git repo; hãy đổi PROJECT_ROOT.')
    remote = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote != REPO_URL:
        raise RuntimeError(f'{PROJECT_ROOT} trỏ tới repo khác: {remote}')
    changes = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'status', '--porcelain'], text=True)
    if changes.strip():
        raise RuntimeError(f'{PROJECT_ROOT} có thay đổi chưa lưu; hãy đổi PROJECT_ROOT hoặc xử lý chúng trước.')
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--depth', '1', 'origin', CODE_COMMIT], check=True)
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
print('M2 code commit:', subprocess.check_output(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(PROJECT_ROOT / 'src/m2_parsing/requirements.txt')], check=True)


In [ ]:
cmd = [sys.executable, '-u', '-m', 'src.m2_parsing.drive_runner',
       '--ranges', *map(str, RANGES), '--m1-parent-id', M1_PARENT_ID,
       '--output', str(OUTPUT), '--workspace', str(WORKSPACE),
       '--workers', str(WORKERS)]
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
print('Kết quả:', OUTPUT)


Kết quả mỗi range nằm trong `OUTPUT/range_N`. Mỗi range mặc định cần đủ 5.000 bài trong checkpoint M1. Khi chạy lại, M2 kiểm tra source, code và dữ liệu trước khi dùng lại kết quả; nếu code hoặc checkpoint đã đổi, chọn thư mục `OUTPUT` mới. Kết quả Rev13/FIX143 cũ được giữ riêng.
